[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/08_landuse-data/08_01_land_cover_impervious.ipynb)


# Module 8, Lesson 1: Annual NLCD Land Cover and Impervious Surface
## Watershed summaries and change analysis with free USGS web services

### Welcome!
This module teaches you how to work with official Annual NLCD land cover and impervious surface data for watershed hydrology. You will learn the difference between a categorical land cover raster and a continuous percent-impervious raster, calculate watershed summaries correctly, and compare two years from one consistent annual product series. The notebook fetches its own raster data from a free USGS web service, so there is nothing to download or upload beyond the watershed boundary file.

### What You'll Work Through Today:
- Understand the difference between categorical and continuous rasters
- Discover the years available in Annual NLCD Collection 1
- Fetch free Annual NLCD rasters for a watershed from the official USGS service
- Validate each response as a real, aligned 30 meter raster
- Clip Annual NLCD land cover and summarize classes by area and percent
- Calculate a defensible watershed-average percent impervious value
- Use the Annual NLCD descriptor to separate roads from other built surfaces
- Compare 2001 and 2025 using one consistent annual methodology
- Export traceable summary tables and clipped rasters

### Module Structure:
1. **Mental Models** - Land cover, land use, and categorical versus continuous rasters
2. **The Data Source** - Annual NLCD and free access options
3. **Workspace Setup** - Libraries and the watershed boundary
4. **Watershed, Data Fetch, and CRS** - Year discovery, tiled WCS requests, and QA/QC
5. **Land Cover Workflow** - Clip, count, and summarize by class
6. **Percent Impervious Workflow** - Correct NoData handling and area weighting
7. **Impervious Descriptor** - Roads versus other built surfaces
8. **Change Analysis** - 2001 versus 2025
9. **Export** - Model-ready tables and clipped rasters
10. **Beyond Annual NLCD** - Coastal and local data refinements

### Prerequisites
This module assumes you have completed Module 1, Module 3, and Module 4. Module 7 is helpful background because its soil outputs pair with land cover in curve number workflows, but it is not required.


## Using AI in This Module

This module works with two very different kinds of rasters in the same lesson: a categorical land cover raster and a continuous percent-impervious raster. If a step feels confusing, ask your AI assistant:

- *"Why can't I average land cover class codes the way I would average a percentage?"*
- *"I clipped a raster to my watershed and the pixel count looks wrong. What should I check?"*
- *"My percent-impervious result changed a lot after I fixed my NoData value. Why would that happen?"*

As always, review what the AI suggests before you run it. This module in particular is built around a real mistake that is easy to make silently. Use your assistant to help you reason through why the mistake happens, not just to get an answer.


## Part 1: Mental Models - Land Cover, Land Use, and Two Kinds of Rasters 🧠

### Land Cover Is Not the Same as Land Use

**Land cover** is what physically covers the ground: forest, pavement, water, grass. **Land use** is how people use that land: a park and a residential yard can have the exact same grass land cover but very different land use. H&H modeling almost always needs land cover, because runoff and infiltration respond to what is physically on the ground, not to zoning. Every product in this module is a land cover product.

### A Quick Raster Recap

Module 4 introduced rasters as grids of cells, each holding a value, with a resolution, a coordinate reference system, and a NoData convention. This module builds directly on that. If any of those words feel unfamiliar, a quick look back at Module 4 will help before continuing here.

### Categorical vs. Continuous Rasters: The Idea This Module Is Built On

This is the single most important distinction in this module, so it is worth sitting with for a moment.

- A **categorical raster** stores a class code in every cell. NLCD land cover is categorical: a cell with the value 42 means "evergreen forest." The number 42 is a label, not a quantity. Averaging class codes together (adding 42 and 21 and dividing by two) produces a meaningless number. You can only count how many cells fall into each class.
- A **continuous raster** stores a measured quantity in every cell. The NLCD fractional impervious surface product is continuous: a cell with the value 37 means an estimated 37 percent of that 30 meter cell is impervious. Averaging these values together is exactly the right thing to do; that is what the product is for.

| | Categorical | Continuous |
|---|---|---|
| Example in this module | Land Cover, Impervious Descriptor | Fractional Impervious Surface |
| What a cell value means | A class code | A measured percentage |
| Correct summary | Count cells per class, convert to area | Average the values, weighted by area |
| Wrong summary | Averaging the codes | Treating it like a category |

### The Three NLCD Products Used in This Module

| Product | Type | What It Tells You |
|---|---|---|
| Land Cover | Categorical | Which land cover class each cell belongs to (forest, developed, water, and so on) |
| Fractional Impervious Surface | Continuous | The estimated percent impervious surface in each cell |
| Impervious Descriptor | Categorical | Whether an impervious cell is a road, or another kind of built surface |

The rule of thumb this module follows: use Land Cover for class summaries, use Fractional Impervious Surface whenever you need a percent impervious number, and use the Impervious Descriptor only when you specifically need to separate roads from buildings. Do not estimate percent impervious just by counting developed land cover classes; a "Developed, Open Space" cell is nowhere near 100 percent impervious, and the fractional product exists precisely so you do not have to guess.

### Why This Matters in H&H Work

Land cover and imperviousness feed directly into:

- **Curve numbers.** Module 4 built a composite curve number assuming a single hydrologic soil group. A real curve number workflow combines land cover (this module) with hydrologic soil group (Module 7) for each pixel.
- **Manning's roughness.** Land cover class is a common basis for assigning roughness zones in a hydraulic model.
- **HEC-HMS and HEC-RAS parameters.** Percent impervious is a direct input to many urban hydrology loss methods.
- **Existing versus future comparisons.** Land cover from two different years is exactly how a "what changed" analysis is built, which this module demonstrates directly.

### Key Terms

| Term | Meaning |
|---|---|
| Class code | A whole number representing one land cover category (categorical rasters only) |
| Legend | The lookup table connecting each class code to a name and a display color |
| Percent impervious | The estimated fraction of a cell's area covered by impervious surface, 0 to 100 |
| NoData | A raster's way of marking cells with no valid value; the correct NoData value differs by product, as this module will show |
| Equal-area projection | A coordinate system where every cell represents the same true ground area, which is what makes area and percentage math meaningful |
| Cell area | For a 30 meter NLCD cell: 30 x 30 = 900 square meters, which is 0.2224 acres |


## Part 2: The Data Source - Annual NLCD and Free Access

### About Annual NLCD

Annual National Land Cover Database Collection 1 is the current USGS land cover product for the conterminous United States. Collection 1.2 provides one 30 meter raster for every year from 1985 through 2025. The six-product suite includes Land Cover, Land Cover Change, Land Cover Confidence, Fractional Impervious Surface, Impervious Descriptor, and Spectral Change Day of Year.

Annual NLCD uses one consistent modeling framework across the time series. That makes comparisons such as 2001 versus 2025 more defensible than mixing separate legacy releases produced with different methods.

The official citation used in this lesson is:

> U.S. Geological Survey, 2024, Annual National Land Cover Database Collection 1 Science Products: U.S. Geological Survey data release, https://doi.org/10.5066/P94UXNTS.

### How This Notebook Gets the Data for Free

USGS distributes Annual NLCD through several systems. Full national tiles are available through EarthExplorer, ScienceBase, the MRLC download tools, and cloud storage. This notebook uses the time-enabled USGS EROS Web Coverage Service that supports the MRLC viewer.

The WCS is anonymous and free. It can return a small GeoTIFF for one year and one watershed bounding box, so students do not need an AWS account, billing information, or an API key.

The workflow follows five safeguards:

1. Discover available years from the live service.
2. Snap requests to the native 30 meter Annual NLCD grid.
3. Split large requests into manageable tiles.
4. Open every response with Rasterio and verify CRS, shape, resolution, NoData, and value range.
5. Use a local course copy first when it is available, then the live WCS, then the GitHub copy as a final fallback.

### Access Options

| Access Method | Best For |
|---|---|
| This lesson's USGS WCS workflow | Free, scripted watershed-sized requests by year |
| MRLC Viewer and Mosaic Download | Manual area-of-interest downloads |
| EarthExplorer | Official tiled archive downloads |
| ScienceBase | Citable, versioned releases |
| USGS cloud storage | Large cloud workflows that already have AWS access |
| WMS | Visualization in GIS software, not analysis of raw values |

### Today's Engineering Scenario

You are assessing development impacts in the Crow Creek area south of Cheyenne, Wyoming. A stormwater master plan needs a current watershed land cover summary, equivalent impervious area, and a consistent historical comparison. We will compare Annual NLCD 2001 with Annual NLCD 2025, the latest year in Collection 1.2.


## Part 3: Workspace Setup 🛠️

Google Colab already includes `rasterio`, `geopandas`, `numpy`, `pandas`, `requests`, and `matplotlib`, so nothing needs to be installed.


In [ ]:
# Tabular and numerical data
import datetime
import math
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

# Spatial data
import geopandas as gpd
import rasterio
import rasterio.mask
from rasterio.io import MemoryFile
from rasterio.transform import from_origin

# Web requests
import requests

# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

print("Libraries imported. Ready to work with Annual NLCD.")


### Uploading the Watershed Boundary

Upload the same watershed boundary file used in Modules 3, 5, and 7. The five Annual NLCD rasters are obtained automatically in Part 4.


In [ ]:
from google.colab import files

print("Please upload this file:")
print("1. NHD__Watershed_Boundaries_HUC_12_Selected.zip")
print()
print("Click 'Choose Files' below and select it.")

uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} file(s):")
for filename in uploaded.keys():
    print(f"   - {filename}")


## Part 4: Watershed, Data Fetch, and CRS 🌍

### Load the Watershed

This module reuses the same Wyoming HUC-12 watershed from Modules 3, 5, and 7: watershed `101900090108`, "Town of South Greeley."


In [ ]:
# Read the watershed shapefile directly from the ZIP archive
watersheds = gpd.read_file(
    'zip://NHD__Watershed_Boundaries_HUC_12_Selected.zip'
)

# Select the HUC-12 used throughout this lesson
target_huc = '101900090108'
target_watershed = watersheds[
    watersheds['HUC12'].astype(str) == target_huc
].copy()

if target_watershed.empty:
    raise RuntimeError(f"HUC-12 {target_huc} was not found in the uploaded file.")
if target_watershed.crs is None:
    raise RuntimeError("The watershed file has no CRS. Confirm the ZIP includes its .prj file.")

print(f"Target watershed: {target_watershed['Name'].iloc[0]}")
print(f"Reported area: {target_watershed['AreaAcres'].iloc[0]:,.0f} acres")
print(f"Watershed CRS: {target_watershed.crs}")


### Fetching Annual NLCD Rasters for This Watershed

Annual NLCD is served in EPSG:5070, an Albers equal-area projection. Reproject the watershed into that CRS before building the request bounds. We use Annual NLCD 2001 as the baseline and 2025 as the current condition. Both years come from Collection 1.2.


In [ ]:
# Pin the analysis years so the lesson remains reproducible
BASELINE_YEAR = 2001
CURRENT_YEAR = 2025

# Native Annual NLCD grid properties from the service metadata
NLCD_CRS = "EPSG:5070"
NLCD_CELL_SIZE = 30.0
ANNUAL_NLCD_NODATA = 250
NLCD_GRID_ANCHOR = 15.0

# Reproject the watershed before requesting data in NLCD coordinates
target_bbox_crs = target_watershed.to_crs(NLCD_CRS)
minx, miny, maxx, maxy = target_bbox_crs.total_bounds

# A small buffer keeps the watershed away from the fetched raster edge
buffer_m = 500
request_bounds = (
    minx - buffer_m,
    miny - buffer_m,
    maxx + buffer_m,
    maxy + buffer_m,
)

print(f"Analysis years: {BASELINE_YEAR} and {CURRENT_YEAR}")
print(f"Watershed bounding box in {NLCD_CRS}, with a {buffer_m} m buffer:")
print(f"  X: {request_bounds[0]:.0f} to {request_bounds[2]:.0f}")
print(f"  Y: {request_bounds[1]:.0f} to {request_bounds[3]:.0f}")


### Reusable Annual NLCD Fetch Helpers

The code below adapts the tested approach used by the `nlcd.py` data-preparation package. WCS 2.0.1 is used only to read the time axis. Raster data are requested with WCS 1.0.0 because that endpoint returns reliable time-selected GeoTIFFs from the image mosaic.

The helper is intentionally strict. A successful HTTP status is not enough because a map service can return an XML error page with status 200. Each response must open as a one-band EPSG:5070 raster with the expected grid, NoData value, and product value range.


In [ ]:
# One time-enabled WCS workspace and layer for each Annual NLCD product
ANNUAL_NLCD_SERVICES = {
    'land_cover': {
        'workspace': 'mrlc_Land-Cover-Native_conus_year_data',
        'layer': 'Land-Cover-Native_conus_year_data',
        'allowed_values': {
            11, 12, 21, 22, 23, 24, 31, 41, 42, 43,
            52, 71, 81, 82, 90, 95,
        },
    },
    'impervious': {
        'workspace': 'mrlc_Fractional-Impervious-Surface-Native_conus_year_data',
        'layer': 'Fractional-Impervious-Surface-Native_conus_year_data',
        'value_range': (0, 100),
    },
    'descriptor': {
        'workspace': 'mrlc_Impervious-Descriptor-Native_conus_year_data',
        'layer': 'Impervious-Descriptor-Native_conus_year_data',
        'allowed_values': {0, 1, 2},
    },
}

FALLBACK_BASE_URL = (
    "https://raw.githubusercontent.com/mohsennasab/python-fundamentals-hh/"
    "main/notebooks/08_landuse-data/data"
)

# Keep each public-service request modest. This watershed needs one tile.
WCS_TILE_CELLS = 1024
WCS_RETRIES = 3


def annual_nlcd_wcs_url(product):
    '''Return the official USGS WCS endpoint for one product.'''
    workspace = ANNUAL_NLCD_SERVICES[product]['workspace']
    return f"https://dmsdata.cr.usgs.gov/geoserver/{workspace}/wcs"


def available_annual_nlcd_years():
    '''Read available years from the service, with an offline fallback.'''
    service = ANNUAL_NLCD_SERVICES['land_cover']
    coverage_id = f"{service['workspace']}__{service['layer']}"
    try:
        response = requests.get(
            annual_nlcd_wcs_url('land_cover'),
            params={
                'service': 'WCS',
                'version': '2.0.1',
                'request': 'DescribeCoverage',
                'coverageId': coverage_id,
            },
            timeout=30,
        )
        response.raise_for_status()
        years = sorted({
            int(year)
            for year in re.findall(
                r'<gml:timePosition>(\d{4})', response.text
            )
        })
        if years:
            return years
    except requests.exceptions.RequestException as error:
        print(f"  live year discovery was unavailable ({error})")

    # Collection 1.2 covers 1985 through 2025
    return list(range(1985, 2026))


def align_to_annual_nlcd_grid(bounds):
    '''Snap bounds outward to native 30 meter Annual NLCD cell edges.'''
    minx, miny, maxx, maxy = bounds
    anchor = NLCD_GRID_ANCHOR
    size = NLCD_CELL_SIZE
    minx = anchor + math.floor((minx - anchor) / size) * size
    miny = anchor + math.floor((miny - anchor) / size) * size
    maxx = anchor + math.ceil((maxx - anchor) / size) * size
    maxy = anchor + math.ceil((maxy - anchor) / size) * size
    width = int(round((maxx - minx) / size))
    height = int(round((maxy - miny) / size))
    transform = from_origin(minx, maxy, size, size)
    return (minx, miny, maxx, maxy), transform, width, height


def validate_annual_nlcd_raster(path, product, required_bounds=None):
    '''Confirm a file is a usable Annual NLCD raster for this request.'''
    expected_crs = rasterio.crs.CRS.from_string(NLCD_CRS)
    service = ANNUAL_NLCD_SERVICES[product]

    with rasterio.open(path) as src:
        problems = []
        if src.count != 1:
            problems.append(f"expected 1 band, found {src.count}")
        if src.width < 1 or src.height < 1:
            problems.append("raster has no cells")
        if src.crs != expected_crs:
            problems.append(f"expected {expected_crs}, found {src.crs}")
        if not np.allclose(src.res, (NLCD_CELL_SIZE, NLCD_CELL_SIZE), atol=0.01):
            problems.append(f"expected 30 meter cells, found {src.res}")
        if src.nodata != ANNUAL_NLCD_NODATA:
            problems.append(
                f"expected NoData={ANNUAL_NLCD_NODATA}, found {src.nodata}"
            )

        if required_bounds is not None:
            req_minx, req_miny, req_maxx, req_maxy = required_bounds
            tolerance = NLCD_CELL_SIZE
            if (
                src.bounds.left > req_minx + tolerance
                or src.bounds.bottom > req_miny + tolerance
                or src.bounds.right < req_maxx - tolerance
                or src.bounds.top < req_maxy - tolerance
            ):
                problems.append("raster does not cover the requested bounds")

        values = src.read(1)
        valid_values = values[values != ANNUAL_NLCD_NODATA]
        if valid_values.size == 0:
            problems.append("raster contains no mapped cells")
        elif 'allowed_values' in service:
            unexpected = set(np.unique(valid_values)) - service['allowed_values']
            if unexpected:
                problems.append(f"unexpected class values {sorted(unexpected)}")
        else:
            valid_min, valid_max = service['value_range']
            if valid_values.min() < valid_min or valid_values.max() > valid_max:
                problems.append(
                    f"values fall outside {valid_min} to {valid_max}"
                )

        if problems:
            raise ValueError("; ".join(problems))

        return src.width, src.height, src.crs, src.res


def fetch_wcs_tile(product, year, bounds, width, height):
    '''Request and validate one Annual NLCD tile in memory.'''
    service = ANNUAL_NLCD_SERVICES[product]
    params = {
        'service': 'WCS',
        'version': '1.0.0',
        'request': 'GetCoverage',
        'coverage': f"{service['workspace']}:{service['layer']}",
        'bbox': ','.join(str(value) for value in bounds),
        'crs': NLCD_CRS,
        'response_crs': NLCD_CRS,
        'format': 'GeoTIFF',
        'width': str(width),
        'height': str(height),
        'TIME': f'{year}-01-01',
    }

    last_error = None
    for attempt in range(1, WCS_RETRIES + 1):
        try:
            response = requests.get(
                annual_nlcd_wcs_url(product),
                params=params,
                timeout=180,
            )
            response.raise_for_status()

            content_type = response.headers.get('content-type', '').lower()
            if 'tif' not in content_type and 'image' not in content_type:
                raise RuntimeError(
                    f"unexpected response type {content_type}: "
                    f"{response.text[:200]}"
                )

            # Opening the bytes is the decisive check that this is a raster
            with MemoryFile(response.content) as memory_file:
                with memory_file.open() as src:
                    if src.count != 1:
                        raise RuntimeError(f"expected 1 band, found {src.count}")
                    if src.crs is None or src.crs.to_epsg() != 5070:
                        raise RuntimeError(f"unexpected CRS {src.crs}")
                    if src.shape != (height, width):
                        raise RuntimeError(
                            f"expected {(height, width)}, found {src.shape}"
                        )
                    if src.nodata != ANNUAL_NLCD_NODATA:
                        raise RuntimeError(
                            f"expected NoData=250, found {src.nodata}"
                        )
                    data = src.read(1)
                    tile_transform = src.transform
                    try:
                        colormap = src.colormap(1)
                    except ValueError:
                        colormap = None
            return data, tile_transform, colormap
        except (
            requests.exceptions.RequestException,
            rasterio.errors.RasterioIOError,
            RuntimeError,
        ) as error:
            last_error = error
            print(f"    tile attempt {attempt} failed ({error})")

    raise RuntimeError(
        f"Could not download the {product} tile after {WCS_RETRIES} attempts. "
        f"Details: {last_error}"
    )


def download_annual_nlcd_raster(filename, product, year, bounds):
    '''Download aligned tiles, build one mosaic, and save a GeoTIFF.'''
    aligned_bounds, transform, width, height = align_to_annual_nlcd_grid(bounds)
    minx, miny, maxx, maxy = aligned_bounds
    mosaic = np.full(
        (height, width), ANNUAL_NLCD_NODATA, dtype=np.uint8
    )
    colormap = None

    tiles = []
    for row_offset in range(0, height, WCS_TILE_CELLS):
        for col_offset in range(0, width, WCS_TILE_CELLS):
            tile_height = min(WCS_TILE_CELLS, height - row_offset)
            tile_width = min(WCS_TILE_CELLS, width - col_offset)
            tile_minx = minx + col_offset * NLCD_CELL_SIZE
            tile_maxx = tile_minx + tile_width * NLCD_CELL_SIZE
            tile_maxy = maxy - row_offset * NLCD_CELL_SIZE
            tile_miny = tile_maxy - tile_height * NLCD_CELL_SIZE
            tiles.append((
                row_offset, col_offset, tile_width, tile_height,
                (tile_minx, tile_miny, tile_maxx, tile_maxy),
            ))

    print(
        f"  requesting {width} x {height} cells in "
        f"{len(tiles)} tile(s) from USGS"
    )
    for tile_number, tile in enumerate(tiles, start=1):
        row_offset, col_offset, tile_width, tile_height, tile_bounds = tile
        data, tile_transform, tile_colormap = fetch_wcs_tile(
            product, year, tile_bounds, tile_width, tile_height
        )

        # Place each tile from its georeferencing, not just loop position
        col_start = int(round(
            (tile_transform.c - transform.c) / NLCD_CELL_SIZE
        ))
        row_start = int(round(
            (transform.f - tile_transform.f) / NLCD_CELL_SIZE
        ))
        mosaic[
            row_start:row_start + data.shape[0],
            col_start:col_start + data.shape[1],
        ] = data
        if colormap is None and tile_colormap is not None:
            colormap = tile_colormap
        print(f"    validated tile {tile_number} of {len(tiles)}")

    with rasterio.open(
        filename,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=rasterio.uint8,
        crs=NLCD_CRS,
        transform=transform,
        nodata=ANNUAL_NLCD_NODATA,
        compress='lzw',
    ) as dst:
        dst.write(mosaic, 1)
        if colormap is not None:
            dst.write_colormap(1, colormap)


def fetch_annual_nlcd_raster(filename, product, year, bounds):
    '''Prepare one raster using local, live WCS, then GitHub sources.'''
    output_path = Path(filename)
    aligned_bounds, _, _, _ = align_to_annual_nlcd_grid(bounds)

    # Local-first behavior makes cloned course repositories work offline
    local_candidates = [
        output_path,
        Path('data') / filename,
        Path('notebooks/08_landuse-data/data') / filename,
    ]
    for candidate in local_candidates:
        if not candidate.exists():
            continue
        try:
            validate_annual_nlcd_raster(
                candidate, product, required_bounds=aligned_bounds
            )
            if candidate.resolve() != output_path.resolve():
                shutil.copyfile(candidate, output_path)
            print(f"  using validated local file: {candidate}")
            return
        except (
            OSError,
            rasterio.errors.RasterioIOError,
            ValueError,
        ) as error:
            print(f"  local candidate {candidate} was not usable ({error})")

    # Request the official service when no suitable local file exists
    try:
        download_annual_nlcd_raster(
            filename, product, year, bounds
        )
        width, height, crs, resolution = validate_annual_nlcd_raster(
            output_path, product, required_bounds=aligned_bounds
        )
        print(
            f"  fetched and validated from USGS WCS "
            f"({width} x {height}, {crs}, {resolution})"
        )
        return
    except (
        requests.exceptions.RequestException,
        rasterio.errors.RasterioIOError,
        RuntimeError,
        ValueError,
    ) as error:
        print(f"  live WCS request failed ({error})")

    # A GitHub copy keeps the fixed course example runnable during outages
    fallback_url = f"{FALLBACK_BASE_URL}/{filename}"
    print("  trying the course repository fallback...")
    try:
        response = requests.get(fallback_url, timeout=60)
        response.raise_for_status()
        output_path.write_bytes(response.content)
        width, height, crs, resolution = validate_annual_nlcd_raster(
            output_path, product, required_bounds=aligned_bounds
        )
        print(
            f"  fetched and validated from GitHub "
            f"({width} x {height}, {crs}, {resolution})"
        )
    except (
        OSError,
        requests.exceptions.RequestException,
        rasterio.errors.RasterioIOError,
        ValueError,
    ) as error:
        raise RuntimeError(
            f"No usable source was available for {filename}. Check the "
            "internet connection or confirm the local course file is present."
        ) from error


print("Annual NLCD fetch helpers are ready.")


### Fetching the Five Analysis Rasters

We need land cover and fractional impervious surface for both years, plus the current-year impervious descriptor. The filenames include `annual_nlcd` so they cannot be confused with the older legacy NLCD products.


In [ ]:
# Discover the service time axis and confirm both selected years exist
available_years = available_annual_nlcd_years()
print(
    f"Annual NLCD years available: {available_years[0]} "
    f"through {available_years[-1]}"
)

for selected_year in [BASELINE_YEAR, CURRENT_YEAR]:
    if selected_year not in available_years:
        raise RuntimeError(
            f"Annual NLCD year {selected_year} is not available."
        )

# filename -> (product name used by the helper, year)
nlcd_products = {
    f'annual_nlcd_{BASELINE_YEAR}_land_cover.tif':
        ('land_cover', BASELINE_YEAR),
    f'annual_nlcd_{CURRENT_YEAR}_land_cover.tif':
        ('land_cover', CURRENT_YEAR),
    f'annual_nlcd_{BASELINE_YEAR}_impervious.tif':
        ('impervious', BASELINE_YEAR),
    f'annual_nlcd_{CURRENT_YEAR}_impervious.tif':
        ('impervious', CURRENT_YEAR),
    f'annual_nlcd_{CURRENT_YEAR}_impervious_descriptor.tif':
        ('descriptor', CURRENT_YEAR),
}

print("\nPreparing Annual NLCD rasters for this watershed's extent...")
for filename, (product, year) in nlcd_products.items():
    print(f"\n{filename}:")
    fetch_annual_nlcd_raster(
        filename, product, year, request_bounds
    )

print("\nAll five Annual NLCD rasters are ready.")


### Check the Raster's CRS

Open the current-year Annual NLCD land cover raster and inspect its coordinate reference system, resolution, transform-derived cell area, and NoData value before analysis.


In [ ]:
# Inspect the current land cover raster before using it
lc_current_path = f'annual_nlcd_{CURRENT_YEAR}_land_cover.tif'

with rasterio.open(lc_current_path) as src:
    lc_crs = src.crs
    lc_res = src.res
    lc_nodata = src.nodata
    lc_source_transform = src.transform

# The affine determinant gives true cell area from raster metadata
cell_area_m2 = abs(
    lc_source_transform.a * lc_source_transform.e
    - lc_source_transform.b * lc_source_transform.d
)
cell_area_acres = cell_area_m2 / 4046.86

print(f"Land cover raster CRS: {lc_crs}")
print(f"Resolution: {lc_res[0]:.2f} x {lc_res[1]:.2f} meters")
print(f"Cell area from raster metadata: {cell_area_m2:,.0f} square meters")
print(f"Declared NoData value: {lc_nodata}")


### Why NLCD Uses an Equal-Area Projection

The land cover raster is in a Conterminous US Albers Equal-Area projection, not the geographic latitude/longitude system. This is deliberate. In an equal-area projection, every cell covers the same true ground area (900 square meters for a 30 meter NLCD cell), everywhere in the country. That is exactly what area and percentage calculations need. A geographic CRS would distort cell area depending on latitude, which would quietly bias every area calculation in this module.

### Reproject the Vector, Not the Raster

The watershed boundary is in a different CRS than the land cover raster. There are two ways to fix this, and only one of them is a good idea:

- **Reproject the vector watershed boundary to match the raster's CRS.** This is exact. A polygon boundary is a mathematical shape, and reprojecting it just recalculates its coordinates.
- **Reproject the raster to match the vector's CRS.** This resamples every pixel, which for a categorical raster like land cover means class codes get blended or reassigned at cell edges. For a percent-impervious raster it introduces a similar distortion. Either way, you would be changing the data to avoid changing the boundary, which is backwards.

This module always reprojects vectors to match the raster CRS, never the other way around.

🤖 **Try asking your AI assistant:** *"Why is it better to reproject a watershed boundary to match a raster's CRS, instead of reprojecting the raster to match the vector? What actually happens to the data in each case?"*


In [ ]:
# Reproject the watershed boundary to match the land cover raster's CRS
target_reprojected = target_watershed.to_crs(lc_crs)

print(f"Watershed CRS is now: {target_reprojected.crs}")

# Recompute area from the reprojected polygon, in acres and square miles
# EPSG:5070 uses meters, so area comes back in square meters
area_m2 = target_reprojected.geometry.iloc[0].area
area_acres = area_m2 / 4046.86
area_sqmi = area_acres / 640

print(f"Watershed area: {area_acres:,.0f} acres ({area_sqmi:.1f} square miles)")


## Part 5: Current Land Cover - From Pixels to a Class Table


In [ ]:
# Rasterio needs the watershed geometry in the raster's CRS
watershed_geom = [
    target_reprojected.geometry.iloc[0].__geo_interface__
]

# Clip the current Annual NLCD land cover raster
with rasterio.open(lc_current_path) as src:
    lc_clipped, lc_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )

lc_array = lc_clipped[0]
print(f"Clipped array shape: {lc_array.shape}")
print(f"Total cells in clipped array: {lc_array.size}")


### Step 2: Count Pixels by Class

This raster is categorical, so counting is the correct operation. `numpy.unique` with `return_counts=True` gives every class code present and how many cells hold it. Class code 0 is NLCD's NoData value; no real land cover class uses 0, so it is safe to exclude it here.


In [ ]:
# Count each categorical class, excluding only the documented background
class_values, pixel_counts = np.unique(
    lc_array[lc_array != ANNUAL_NLCD_NODATA],
    return_counts=True,
)

print(f"Found {len(class_values)} land cover classes in the watershed")
for code_value, count in zip(class_values, pixel_counts):
    print(f"  code {code_value}: {count} pixels")


### Step 3: The Annual NLCD Legend

Annual NLCD uses the familiar modified Anderson Level II class codes. The dictionary below connects each raster value to a name and official display color.


In [ ]:
# Official Annual NLCD land cover class names
nlcd_classes = {
    11: 'Open Water',
    12: 'Perennial Ice/Snow',
    21: 'Developed, Open Space',
    22: 'Developed, Low Intensity',
    23: 'Developed, Medium Intensity',
    24: 'Developed, High Intensity',
    31: 'Barren Land',
    41: 'Deciduous Forest',
    42: 'Evergreen Forest',
    43: 'Mixed Forest',
    52: 'Shrub/Scrub',
    71: 'Grassland/Herbaceous',
    81: 'Pasture/Hay',
    82: 'Cultivated Crops',
    90: 'Woody Wetlands',
    95: 'Emergent Herbaceous Wetlands',
}

# Official display colors, one per class code
nlcd_colors = {
    11: '#466b9f', 12: '#d1def8', 21: '#dec5c5',
    22: '#d99282', 23: '#eb0000', 24: '#ab0000',
    31: '#b3ac9f', 41: '#68ab5f', 42: '#1c5f2c',
    43: '#b5c58f', 52: '#ccb879', 71: '#dfdfc2',
    81: '#dcd939', 82: '#ab6c28', 90: '#b8d9eb',
    95: '#6c9fb8',
}

print(f"Legend covers {len(nlcd_classes)} classes")


### Step 4: Convert Pixel Counts to Area and Percent

The cell dimensions were read from the raster metadata in Part 4. Multiply them to get cell area, convert that area to acres, then calculate each class as a percentage of the watershed's total classified area. Reading the dimensions from the file keeps this workflow valid if you later use a raster with a different resolution.


In [ ]:
total_pixels = pixel_counts.sum()

lc_summary = pd.DataFrame({
    'nlcd_code': class_values,
    'nlcd_class': [nlcd_classes.get(int(c), f'Unknown ({c})') for c in class_values],
    'pixel_count': pixel_counts,
})

lc_summary['area_acres'] = lc_summary['pixel_count'] * cell_area_acres
lc_summary['percent_watershed'] = 100 * lc_summary['pixel_count'] / total_pixels

lc_summary = lc_summary.sort_values('percent_watershed', ascending=False).reset_index(drop=True)

print(lc_summary[['nlcd_class', 'area_acres', 'percent_watershed']].round(1))


### Step 5: Sanity Check Against the Polygon Area

Compare the total classified area from the raster to the watershed's true polygon area from Part 4. They should be close; small differences come from how pixel edges approximate an irregular boundary.


In [ ]:
raster_total_acres = lc_summary['area_acres'].sum()

print(f"Watershed area from polygon geometry: {area_acres:,.0f} acres")
print(f"Watershed area from classified raster pixels: {raster_total_acres:,.0f} acres")
print(f"Difference: {abs(area_acres - raster_total_acres):,.0f} acres "
      f"({100*abs(area_acres - raster_total_acres)/area_acres:.1f}% of watershed area)")


A small difference here is normal and expected; it comes from approximating a curved, irregular watershed boundary with a grid of square 30 meter cells. A large difference (more than a few percent) would be worth investigating, since it could mean a CRS mismatch or a clipping error.

### Step 6: Map the Clipped Land Cover


In [ ]:
# Build a categorical colormap for the classes present here
present_codes = sorted(class_values.tolist())
colors_in_order = [nlcd_colors[code] for code in present_codes]
cmap = ListedColormap(colors_in_order)

# Convert raw class codes to consecutive plotting positions
code_to_index = {
    code: index for index, code in enumerate(present_codes)
}
lc_display = np.vectorize(
    lambda value: code_to_index.get(value, -1)
)(lc_array)
lc_display = np.ma.masked_where(
    lc_array == ANNUAL_NLCD_NODATA, lc_display
)

fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(
    lc_display,
    cmap=cmap,
    vmin=0,
    vmax=len(present_codes) - 1,
)

legend_patches = [
    Patch(facecolor=nlcd_colors[code], label=nlcd_classes[code])
    for code in present_codes
]
ax.legend(
    handles=legend_patches,
    loc='center left',
    bbox_to_anchor=(1.0, 0.5),
    fontsize=9,
    frameon=True,
)

ax.set_title(
    f"Annual NLCD {CURRENT_YEAR} Land Cover\n"
    f"{target_watershed['Name'].iloc[0]}"
)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()


### What This Output Tells Us

Grassland dominates this watershed, but the developed classes (open space through high intensity) together account for a meaningful share of the area. This is exactly the mixed rural/exurban character that makes this watershed a useful case for a stormwater screening study: it is not fully rural, and it is not fully urban.


## Part 6: Fractional Impervious Surface - A NoData Mistake Worth Knowing

### First, the Wrong Way

The fractional impervious raster is continuous, so the correct watershed summary is an area-weighted average. A tempting first attempt is to treat 0 as missing because 0 often means background in categorical rasters. Run that incorrect approach once so you can see why product documentation and valid-area checks matter.


In [ ]:
imp_current_path = f'annual_nlcd_{CURRENT_YEAR}_impervious.tif'

with rasterio.open(imp_current_path) as src:
    imp_clipped_wrong, _ = rasterio.mask.mask(
        src, watershed_geom, crop=True, nodata=0
    )

imp_array_wrong = imp_clipped_wrong[0]

# This is intentionally wrong: 0 means 0 percent impervious, not NoData
valid_wrong = imp_array_wrong[imp_array_wrong != 0]
print(f"Valid cell count after incorrectly excluding 0: {valid_wrong.size}")
print(f"Incorrect mean percent impervious: {valid_wrong.mean():.2f}%")


### Something Is Wrong

Compare the valid cell count above with the land cover count from Part 5. Both products use the same 30 meter grid and watershed clip, so their valid areas should be nearly the same. Excluding 0 removes every cell with a legitimate value of 0 percent impervious and biases the mean toward developed cells.

### Finding the Real Answer

The Annual NLCD Collection 1.2 user guide and live WCS metadata define **250** as the background value. Fractional impervious values from 0 through 100 are valid percentages. The same background value of 250 applies to the Annual NLCD land cover and impervious descriptor products used here.


In [ ]:
# Repeat the clip with the documented Annual NLCD background value
with rasterio.open(imp_current_path) as src:
    imp_clipped, imp_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )
    imp_source_transform = src.transform

imp_array = imp_clipped[0]
valid_imp_mask = imp_array != ANNUAL_NLCD_NODATA

print(
    f"Background cells inside cropped array: "
    f"{(imp_array == ANNUAL_NLCD_NODATA).sum()}"
)
print(f"Valid impervious cell count: {valid_imp_mask.sum()}")
print(
    f"Valid value range: {imp_array[valid_imp_mask].min()} "
    f"to {imp_array[valid_imp_mask].max()} percent"
)

# Grid alignment is required before comparing or combining products
if (
    imp_array.shape != lc_array.shape
    or not imp_transform.almost_equals(lc_transform)
):
    raise RuntimeError(
        "Land cover and impervious clips are not on the same grid."
    )
print("Land cover and impervious grid alignment QA/QC passed.")


The corrected valid cell count should now agree with the land cover result. Cells with 0 percent impervious remain in the denominator, where they belong.

### The Corrected Calculation


In [ ]:
imp_valid = imp_array[valid_imp_mask].astype(float)

mean_impervious_wrong = valid_wrong.mean()
mean_impervious_correct = imp_valid.mean()

print(
    f"Wrong approach, masked on 0:   "
    f"{mean_impervious_wrong:.2f}% mean impervious"
)
print(
    f"Correct approach, masked on 250: "
    f"{mean_impervious_correct:.2f}% mean impervious"
)
print(
    f"\nThe wrong approach overstated watershed imperviousness by "
    f"{mean_impervious_wrong - mean_impervious_correct:.1f} "
    "percentage points."
)


### Why the Total Area Estimate Looked Fine, but the Percentage Did Not

If you had instead reported the total equivalent impervious acreage (percent times area, summed across all pixels), you might not have noticed a problem: the excluded pixels were mostly near zero, so they barely contributed to that sum either way. It is specifically the **average percentage** that gets distorted, because the denominator (how many pixels you are averaging over) shrank dramatically while the numerator barely changed. This is exactly why it is worth checking more than one derived number, and why a pixel-count mismatch between two supposedly identical clips is worth chasing down instead of ignoring.

### The Full Area-Weighted Calculation

Now that NoData is handled correctly, compute the full area-weighted percent impervious and the equivalent impervious area, following the same explicit, step-by-step style used for Ksat in Module 7.

One thing to notice: because every NLCD cell covers the same 900 square meters, the area-weighted average below comes out exactly equal to the simple mean printed above. So why bother with the longer form? Two reasons. First, it produces the equivalent impervious area as a byproduct, which is a useful engineering number on its own. Second, the explicit form is the one that stays correct when cells do not all represent the same area, for example when combining rasters of different resolutions or weighting partial cells along a boundary. Writing it out now builds the right habit for those situations.


In [ ]:
# Use the impervious raster's own affine transform for its cell area
impervious_cell_area_m2 = abs(
    imp_source_transform.a * imp_source_transform.e
    - imp_source_transform.b * imp_source_transform.d
)

# Each cell contributes its fractional imperviousness times cell area
impervious_area_per_cell_m2 = (
    imp_valid / 100
) * impervious_cell_area_m2

total_impervious_area_m2 = impervious_area_per_cell_m2.sum()
total_impervious_area_acres = total_impervious_area_m2 / 4046.86

total_valid_area_m2 = imp_valid.size * impervious_cell_area_m2
total_valid_area_acres = total_valid_area_m2 / 4046.86

percent_impervious_current = (
    100 * total_impervious_area_m2 / total_valid_area_m2
)

print(f"Total valid area: {total_valid_area_acres:,.0f} acres")
print(
    f"Equivalent impervious area: "
    f"{total_impervious_area_acres:,.0f} acres"
)
print(
    f"Watershed-average percent impervious: "
    f"{percent_impervious_current:.2f}%"
)


🤖 **Try asking your AI assistant:** *"I calculated mean percent impervious for a watershed using Annual NLCD Fractional Impervious Surface. Help me interpret this result for H&H modeling, including limitations related to 30 meter resolution, product year, and local refinement."*

### Engineering Caution: Check NoData, Do Not Assume It

Annual NLCD uses 250 as background for the three products in this lesson. A value of 0 is valid in the fractional impervious raster and the descriptor raster. Keep the product version, its official user guide, and the file metadata together when deciding what to mask.


## Part 7: Impervious Descriptor - Roads or Other Built Surfaces?

The Annual NLCD descriptor is categorical. Its classes are 0 for non-urban or not impervious, 1 for roads, and 2 for urban or other built surfaces. The background value is 250.

Pixel counts describe land coverage, but the hydrologically useful question is how much equivalent impervious area is associated with each descriptor class. We answer that by weighting every descriptor cell by its fractional impervious percentage.


In [ ]:
desc_path = f'annual_nlcd_{CURRENT_YEAR}_impervious_descriptor.tif'

with rasterio.open(desc_path) as src:
    desc_clipped, desc_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )
    descriptor_source_transform = src.transform

desc_array = desc_clipped[0]

# Values can be combined only when both rasters use the same cells
if (
    desc_array.shape != imp_array.shape
    or not desc_transform.almost_equals(imp_transform)
):
    raise RuntimeError(
        "Descriptor and impervious rasters are not on the same grid."
    )

descriptor_cell_area_m2 = abs(
    descriptor_source_transform.a * descriptor_source_transform.e
    - descriptor_source_transform.b * descriptor_source_transform.d
)
combined_valid_mask = (
    (desc_array != ANNUAL_NLCD_NODATA)
    & (imp_array != ANNUAL_NLCD_NODATA)
)
desc_values, desc_counts = np.unique(
    desc_array[combined_valid_mask],
    return_counts=True,
)

descriptor_names = {
    0: 'Non-urban / not impervious',
    1: 'Roads',
    2: 'Urban (other built surfaces)',
}

descriptor_rows = []
total_descriptor_cells = desc_counts.sum()

for code_value, count in zip(desc_values, desc_counts):
    class_mask = combined_valid_mask & (desc_array == code_value)
    class_impervious_area_m2 = (
        (imp_array[class_mask].astype(float) / 100)
        * descriptor_cell_area_m2
    ).sum()
    descriptor_rows.append({
        'descriptor_code': int(code_value),
        'descriptor_class': descriptor_names.get(
            int(code_value), f'code {code_value}'
        ),
        'pixel_count': int(count),
        'percent_mapped_cells': 100 * count / total_descriptor_cells,
        'impervious_area_acres': class_impervious_area_m2 / 4046.86,
    })

descriptor_summary = pd.DataFrame(descriptor_rows)
descriptor_total_impervious_acres = (
    descriptor_summary['impervious_area_acres'].sum()
)
descriptor_summary['percent_total_impervious_area'] = (
    100
    * descriptor_summary['impervious_area_acres']
    / descriptor_total_impervious_acres
)
descriptor_summary = descriptor_summary.sort_values(
    'impervious_area_acres', ascending=False
).reset_index(drop=True)

descriptor_area_difference_pct = (
    100
    * abs(
        descriptor_total_impervious_acres
        - total_impervious_area_acres
    )
    / total_impervious_area_acres
)

print(descriptor_summary[[
    'descriptor_class',
    'percent_mapped_cells',
    'impervious_area_acres',
    'percent_total_impervious_area',
]].round(2).to_string(index=False))
print(
    f"\nDescriptor-weighted impervious area check: "
    f"{descriptor_total_impervious_acres:,.1f} acres"
)
print(f"Difference from Part 6 total: {descriptor_area_difference_pct:.2f}%")

if descriptor_area_difference_pct > 1:
    print(
        "WARNING: Descriptor and fractional impervious totals differ "
        "by more than 1%. Investigate before interpreting the split."
    )
else:
    print("Descriptor-weighted area QA/QC passed.")


### What This Tells Us

`percent_mapped_cells` describes how much mapped land falls in each descriptor category. `percent_total_impervious_area` is the engineering summary. It reports how much equivalent impervious area comes from roads, other built surfaces, or cells classified as non-urban after weighting by the fractional impervious value.

Keep the pixel and weighted columns together. A descriptor class can cover many cells without contributing the same amount of impervious area as a smaller but more intensely developed class.


## Part 8: Annual Change Analysis - 2001 versus 2025

### Land Cover Change

Repeat the Part 5 workflow for the baseline land cover raster. Both years come from Annual NLCD Collection 1.2 and use the same grid and classification method.


In [ ]:
lc_baseline_path = f'annual_nlcd_{BASELINE_YEAR}_land_cover.tif'

with rasterio.open(lc_baseline_path) as src:
    lc_baseline_clipped, lc_baseline_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )

lc_baseline_array = lc_baseline_clipped[0]
if (
    lc_baseline_array.shape != lc_array.shape
    or not lc_baseline_transform.almost_equals(lc_transform)
):
    raise RuntimeError(
        "Baseline and current land cover rasters are not on the same grid."
    )

class_values_baseline, pixel_counts_baseline = np.unique(
    lc_baseline_array[
        lc_baseline_array != ANNUAL_NLCD_NODATA
    ],
    return_counts=True,
)

lc_baseline_summary = pd.DataFrame({
    'nlcd_code': class_values_baseline,
    'nlcd_class': [
        nlcd_classes.get(int(code), f'Unknown ({code})')
        for code in class_values_baseline
    ],
    'pixel_count': pixel_counts_baseline,
    'percent_watershed_baseline': (
        100
        * pixel_counts_baseline
        / pixel_counts_baseline.sum()
    ),
})

# Outer merge keeps classes that occur in only one year
comparison = lc_summary[[
    'nlcd_code', 'percent_watershed'
]].merge(
    lc_baseline_summary[[
        'nlcd_code', 'percent_watershed_baseline'
    ]],
    on='nlcd_code',
    how='outer',
).rename(columns={
    'percent_watershed': 'percent_watershed_current'
})

comparison[[
    'percent_watershed_baseline',
    'percent_watershed_current',
]] = comparison[[
    'percent_watershed_baseline',
    'percent_watershed_current',
]].fillna(0)

comparison['nlcd_class'] = comparison['nlcd_code'].map(
    lambda code: nlcd_classes.get(
        int(code), f'Unknown ({code})'
    )
)
comparison['change_pct_points'] = (
    comparison['percent_watershed_current']
    - comparison['percent_watershed_baseline']
)
comparison = comparison[[
    'nlcd_code',
    'nlcd_class',
    'percent_watershed_baseline',
    'percent_watershed_current',
    'change_pct_points',
]].sort_values(
    'percent_watershed_current', ascending=False
)

print(comparison.round(2).to_string(index=False))


In [ ]:
# QA/QC: confirm both years represent nearly the same mapped area
land_cover_valid_area_check = pd.DataFrame({
    'nlcd_year': [BASELINE_YEAR, CURRENT_YEAR],
    'valid_cell_count': [
        pixel_counts_baseline.sum(),
        total_pixels,
    ],
})
land_cover_valid_area_check['valid_area_acres'] = (
    land_cover_valid_area_check['valid_cell_count']
    * cell_area_acres
)
land_cover_valid_area_check['percent_of_polygon_area'] = (
    100
    * land_cover_valid_area_check['valid_area_acres']
    / area_acres
)

land_cover_valid_area_change_pct = 100 * (
    land_cover_valid_area_check.loc[1, 'valid_area_acres']
    - land_cover_valid_area_check.loc[0, 'valid_area_acres']
) / land_cover_valid_area_check.loc[0, 'valid_area_acres']

print("Land cover valid-area QA/QC:")
print(land_cover_valid_area_check.round(2).to_string(index=False))
if abs(land_cover_valid_area_change_pct) > 1:
    print(
        "WARNING: Valid mapped area differs by more than 1% "
        "between years. Investigate before interpreting change."
    )
else:
    print(
        "Valid-area QA/QC passed: mapped area differs by "
        "no more than 1% between years."
    )


### Fractional Impervious Change

Repeat the corrected Part 6 calculation for the baseline year. Do not mix legacy NLCD and Annual NLCD values in this comparison.


In [ ]:
imp_baseline_path = f'annual_nlcd_{BASELINE_YEAR}_impervious.tif'

with rasterio.open(imp_baseline_path) as src:
    imp_baseline_clipped, imp_baseline_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )

imp_baseline_array = imp_baseline_clipped[0]
if (
    imp_baseline_array.shape != imp_array.shape
    or not imp_baseline_transform.almost_equals(imp_transform)
):
    raise RuntimeError(
        "Baseline and current impervious rasters are not on the same grid."
    )

imp_baseline_valid = imp_baseline_array[
    imp_baseline_array != ANNUAL_NLCD_NODATA
].astype(float)

impervious_area_baseline_m2 = (
    (imp_baseline_valid / 100) * impervious_cell_area_m2
).sum()
valid_area_baseline_m2 = (
    imp_baseline_valid.size * impervious_cell_area_m2
)
percent_impervious_baseline = (
    100 * impervious_area_baseline_m2 / valid_area_baseline_m2
)

impervious_valid_area_check = pd.DataFrame({
    'nlcd_year': [BASELINE_YEAR, CURRENT_YEAR],
    'valid_cell_count': [
        imp_baseline_valid.size,
        imp_valid.size,
    ],
    'valid_area_acres': [
        valid_area_baseline_m2 / 4046.86,
        total_valid_area_acres,
    ],
})
impervious_valid_area_change_pct = 100 * (
    impervious_valid_area_check.loc[1, 'valid_area_acres']
    - impervious_valid_area_check.loc[0, 'valid_area_acres']
) / impervious_valid_area_check.loc[0, 'valid_area_acres']

print("Impervious valid-area QA/QC:")
print(impervious_valid_area_check.round(2).to_string(index=False))
if abs(impervious_valid_area_change_pct) > 1:
    print(
        "WARNING: Valid mapped area differs by more than 1% "
        "between years. Investigate before interpreting change."
    )
else:
    print(
        "Valid-area QA/QC passed: mapped area differs by "
        "no more than 1% between years."
    )

elapsed_years = CURRENT_YEAR - BASELINE_YEAR
print(
    f"{BASELINE_YEAR} watershed-average percent impervious: "
    f"{percent_impervious_baseline:.2f}%"
)
print(
    f"{CURRENT_YEAR} watershed-average percent impervious: "
    f"{percent_impervious_current:.2f}%"
)
print(
    f"Change over {elapsed_years} years: "
    f"{percent_impervious_current - percent_impervious_baseline:+.2f} "
    "percentage points"
)


### Side-by-Side Maps


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

map_arrays = [lc_baseline_array, lc_array]
map_years = [BASELINE_YEAR, CURRENT_YEAR]

for axis, array, year in zip(axes, map_arrays, map_years):
    codes_present = sorted(np.unique(
        array[array != ANNUAL_NLCD_NODATA]
    ).tolist())
    colors_present = [
        nlcd_colors[code] for code in codes_present
    ]
    cmap_year = ListedColormap(colors_present)
    index_lookup = {
        code: index for index, code in enumerate(codes_present)
    }
    display_array = np.vectorize(
        lambda value: index_lookup.get(value, -1)
    )(array)
    display_array = np.ma.masked_where(
        array == ANNUAL_NLCD_NODATA, display_array
    )

    axis.imshow(
        display_array,
        cmap=cmap_year,
        vmin=0,
        vmax=len(codes_present) - 1,
    )
    axis.set_title(f"Annual NLCD {year} Land Cover")
    axis.set_xticks([])
    axis.set_yticks([])

fig.suptitle(
    f"{target_watershed['Name'].iloc[0]}: "
    f"{BASELINE_YEAR} vs. {CURRENT_YEAR}",
    fontsize=14,
)
plt.tight_layout()
plt.show()


### What This Tells Us

The comparison quantifies how the mapped developed footprint and fractional imperviousness changed over 24 years. Because both rasters come from Annual NLCD Collection 1.2, the comparison uses one consistent annual framework.

This remains a screening-level result. Interpret change only after grid alignment and valid-area checks pass, and review unexpected class transitions against imagery, local GIS, and known development history.


## Part 9: Export - Model-Ready Tables and Rasters

Add the product family, collection version, DOI, year, access date, and processing notes to the exported tables. These fields distinguish Annual NLCD from legacy NLCD and make later change calculations traceable.


In [ ]:
access_date = datetime.date.today().isoformat()
collection_version = 'Annual NLCD Collection 1.2'
source_doi = 'https://doi.org/10.5066/P94UXNTS'

# Land cover summary for both years
lc_current_export = lc_summary.copy()
lc_current_export['nlcd_year'] = CURRENT_YEAR

lc_baseline_export = lc_baseline_summary.rename(columns={
    'percent_watershed_baseline': 'percent_watershed',
}).copy()
lc_baseline_export['area_acres'] = (
    lc_baseline_export['pixel_count'] * cell_area_acres
)
lc_baseline_export['nlcd_year'] = BASELINE_YEAR

lc_export_combined = pd.concat(
    [lc_baseline_export, lc_current_export],
    ignore_index=True,
)
lc_export_combined['watershed_id'] = target_huc
lc_export_combined['source'] = 'USGS Annual NLCD Land Cover'
lc_export_combined['collection_version'] = collection_version
lc_export_combined['source_doi'] = source_doi
lc_export_combined['access_date'] = access_date
lc_export_combined['processing_notes'] = (
    'Clipped to watershed; NoData=250 excluded; '
    'cell area read from affine transform'
)

lc_export_combined.to_csv(
    'watershed_land_cover_summary.csv', index=False
)
print(
    f"Saved watershed_land_cover_summary.csv "
    f"({len(lc_export_combined)} rows)"
)

# Fractional impervious summary for both years
impervious_summary = pd.DataFrame([
    {
        'watershed_id': target_huc,
        'nlcd_year': BASELINE_YEAR,
        'mean_impervious_percent': round(
            percent_impervious_baseline, 2
        ),
        'equivalent_impervious_area_acres': round(
            impervious_area_baseline_m2 / 4046.86, 1
        ),
        'total_valid_area_acres': round(
            valid_area_baseline_m2 / 4046.86, 1
        ),
    },
    {
        'watershed_id': target_huc,
        'nlcd_year': CURRENT_YEAR,
        'mean_impervious_percent': round(
            percent_impervious_current, 2
        ),
        'equivalent_impervious_area_acres': round(
            total_impervious_area_acres, 1
        ),
        'total_valid_area_acres': round(
            total_valid_area_acres, 1
        ),
    },
])
impervious_summary['source'] = (
    'USGS Annual NLCD Fractional Impervious Surface'
)
impervious_summary['collection_version'] = collection_version
impervious_summary['source_doi'] = source_doi
impervious_summary['processing_notes'] = (
    'NoData=250 excluded; 0 is valid; impervious-area weighted'
)
impervious_summary['access_date'] = access_date

impervious_summary.to_csv(
    'watershed_impervious_summary.csv', index=False
)
print(
    f"Saved watershed_impervious_summary.csv "
    f"({len(impervious_summary)} rows)"
)

# Current-year descriptor summary
descriptor_summary_export = descriptor_summary.copy()
descriptor_summary_export['watershed_id'] = target_huc
descriptor_summary_export['nlcd_year'] = CURRENT_YEAR
descriptor_summary_export['source'] = (
    'USGS Annual NLCD Impervious Descriptor'
)
descriptor_summary_export['collection_version'] = collection_version
descriptor_summary_export['source_doi'] = source_doi
descriptor_summary_export['processing_notes'] = (
    'NoData=250 excluded; classes weighted by fractional impervious area'
)
descriptor_summary_export['access_date'] = access_date
descriptor_summary_export.to_csv(
    'watershed_impervious_descriptor_summary.csv',
    index=False,
)
print(
    "Saved watershed_impervious_descriptor_summary.csv "
    f"({len(descriptor_summary_export)} rows)"
)

impervious_summary


### Exporting Clipped Rasters

Save the clipped current-year land cover and impervious rasters as GeoTIFFs. Each export keeps the transform from its own clip and uses the Annual NLCD background value of 250.


In [ ]:
clipped_land_cover_filename = (
    f'clipped_annual_nlcd_land_cover_{CURRENT_YEAR}.tif'
)
clipped_impervious_filename = (
    f'clipped_annual_nlcd_impervious_{CURRENT_YEAR}.tif'
)

# Export current land cover with its source color table
with rasterio.open(lc_current_path) as src:
    output_profile = src.profile.copy()
    output_profile.update(
        height=lc_array.shape[0],
        width=lc_array.shape[1],
        transform=lc_transform,
        nodata=ANNUAL_NLCD_NODATA,
        compress='lzw',
    )
    try:
        source_colormap = src.colormap(1)
    except ValueError:
        source_colormap = None

with rasterio.open(
    clipped_land_cover_filename, 'w', **output_profile
) as dst:
    dst.write(lc_array, 1)
    if source_colormap is not None:
        dst.write_colormap(1, source_colormap)

# Export current fractional impervious surface with its own transform
with rasterio.open(imp_current_path) as src:
    output_profile = src.profile.copy()
    output_profile.update(
        height=imp_array.shape[0],
        width=imp_array.shape[1],
        transform=imp_transform,
        nodata=ANNUAL_NLCD_NODATA,
        compress='lzw',
    )

with rasterio.open(
    clipped_impervious_filename, 'w', **output_profile
) as dst:
    dst.write(imp_array, 1)

print(
    f"Saved {clipped_land_cover_filename} and "
    f"{clipped_impervious_filename}"
)


In [ ]:
# Download all lesson outputs to your computer
from google.colab import files

output_files = [
    'watershed_land_cover_summary.csv',
    'watershed_impervious_summary.csv',
    'watershed_impervious_descriptor_summary.csv',
    clipped_land_cover_filename,
    clipped_impervious_filename,
]

for output_file in output_files:
    files.download(output_file)


### Connecting Forward

This module's land cover table and Module 7's hydrologic soil group table are the two inputs a full composite curve number workflow needs. The fractional impervious result can also support HEC-HMS parameter screening, but it is not a calibrated directly connected impervious area. Confirm connectivity, drainage, and local development data before design use.


## Part 10: Beyond Annual NLCD - Coastal and Local Alternatives

### NOAA C-CAP

NOAA C-CAP provides regional and high-resolution land cover products focused on coastal areas. It uses a different classification scheme and coverage. Evaluate it for coastal watersheds where those products better match the project scale.

### Local and Project-Specific Data

Annual NLCD's 30 meter resolution is appropriate for watershed and regional screening. It is too coarse for parcel-scale design. Local municipal impervious layers, building footprints, roadway inventories, and recent imagery may be better for detailed drainage studies.

### Source Hierarchy Summary

| If you need... | Use |
|---|---|
| A consistent annual CONUS land cover and impervious series | Annual NLCD |
| Coastal-specific or higher-resolution coastal land cover | NOAA C-CAP |
| Parcel-scale design detail | Local GIS, imagery, and as-built data |


## Engineering Cautions

1. **Categorical rasters hold codes, not quantities.** Never average land cover or descriptor class codes.
2. **Use fractional imperviousness for percent impervious.** Developed land cover classes include permeable surfaces.
3. **Annual NLCD background is 250 for these products.** Values from 0 to 100 are valid fractional impervious percentages. Descriptor values 0, 1, and 2 are also valid.
4. **Keep grids aligned before cell-by-cell work.** Check CRS, shape, transform, resolution, and valid area.
5. **Reproject vectors to match the raster.** Resampling a categorical raster can change class assignments at boundaries.
6. **Thirty-meter data support screening, not parcel-scale design.**
7. **Annual NLCD is not directly connected impervious area.** Drainage connectivity needs separate engineering judgment.
8. **Document collection version and year.** Annual updates can extend and revise the time series.


## Troubleshooting

| Problem | Likely Cause | What to Try |
|---|---|---|
| Year discovery fails | Temporary service or network issue | The notebook uses the Collection 1.2 year list and continues |
| WCS returns XML instead of a raster | Invalid request or service issue | Read the printed error, then rerun the fetch cell |
| A fetch is slow | The bounding box needs several tiles | Confirm one HUC-12 is selected and reduce `buffer_m` if appropriate |
| Local fallback is rejected | It does not cover the requested watershed or failed validation | Let the helper continue to the live WCS |
| Clipped raster is empty | Watershed is outside CONUS coverage or CRS is wrong | Print watershed and raster CRS and bounds |
| Land cover and impervious grids do not align | Different bounds, transforms, or products were mixed | Rerun all five requests with the same `request_bounds` |
| Percent impervious is too high | Legitimate 0 values were excluded | Use 250 as Annual NLCD background |
| Descriptor totals do not match | Grids, masks, or NoData handling differ | Check alignment and combined valid mask before interpreting |
| Colab upload fails | The watershed ZIP was not selected | Rerun the upload cell and select the course file |


## Practice Exercises

Each exercise can be completed by lightly editing code already used in the lesson.

### Exercise 1: A Different Watershed

Change `target_huc` to another HUC-12 in the course file. Rerun the watershed selection, request bounds, fetch, clip, and summary cells. The local fallback covers the course watersheds, and the live WCS supports other CONUS watersheds when a suitable local file is not present.


In [ ]:
# EXERCISE 1: A different watershed
# Your code here.
#
# Steps:
# 1. Choose another HUC-12 code from the watersheds table.
# 2. Re-select target_watershed and rebuild request_bounds.
# 3. Call fetch_annual_nlcd_raster() for the five products.
# 4. Reproject the watershed to the raster CRS.
# 5. Rerun the clip, area, and valid-coverage checks.


### Exercise 2: Impervious Hotspots

Using `imp_array`, count valid cells above 50 percent impervious and convert their full cell footprint to acres. Explain why this hotspot footprint is different from equivalent impervious area.


In [ ]:
# EXERCISE 2: Impervious hotspots
# Your code here.
#
# Steps:
# 1. Build a mask for values greater than 50 and not equal to 250.
# 2. Count cells meeting both conditions.
# 3. Multiply by cell_area_acres for hotspot footprint area.
# 4. Explain why this is not equivalent impervious area.


### Exercise 3: Road Share of Impervious Area

Use `descriptor_summary` to report the percentage of equivalent impervious area assigned to roads, code 1, versus urban or other built surfaces, code 2. Keep code 0 visible as a QA/QC category.


In [ ]:
# EXERCISE 3: Descriptor road percentage
# Your code here.
#
# Steps:
# 1. Select the row where descriptor_code equals 1 for roads.
# 2. Select descriptor_code 2 for other built surfaces.
# 3. Report percent_total_impervious_area for both rows.
# 4. Check whether code 0 contributes meaningful impervious area.


### Challenge Exercise: Choose Another Annual Year

Change `CURRENT_YEAR` to another available year and rerun the fetch and analysis workflow. Use a filename that includes the new year. Compare the new result with 2025 and explain why collection version, valid area, and grid alignment still need to be documented.

Suggested prompt:

*"Help me adapt my Annual NLCD notebook so I can compare any two years returned by available_annual_nlcd_years(). Keep land cover categorical, keep fractional impervious continuous, use NoData 250, and require grid and valid-area checks before calculating change."*


In [ ]:
# CHALLENGE EXERCISE: choose another Annual NLCD year
# Your code here.
#
# Steps:
# 1. Print available_years and choose a different year.
# 2. Build year-specific filenames for land cover and imperviousness.
# 3. Fetch both products with fetch_annual_nlcd_raster().
# 4. Clip them to the watershed and verify grid alignment.
# 5. Compare class percentages and mean imperviousness with 2025.


## That's Module 8 Done!

You used free, anonymous USGS web services to discover and retrieve Annual NLCD data without an AWS account. You validated the raster responses, kept categorical and continuous products on separate calculation paths, caught a NoData mistake, summarized impervious descriptors by impervious-area weighting, and compared two years from one consistent annual product series.

### What you can do now

- Fetch Annual NLCD land cover and impervious products for a CONUS watershed
- Verify CRS, resolution, transform, NoData, value range, and valid area
- Summarize categorical land cover by area
- Calculate watershed-average fractional imperviousness and equivalent impervious area
- Separate road and other built contributions using impervious-area weighting
- Compare any two Annual NLCD years with clear QA/QC

### Next Steps

- Combine Annual NLCD land cover with Module 7 hydrologic soil groups for curve number workflows.
- Compare the screening results with local impervious and development data.
- Use local or higher-resolution sources when project scale requires more detail.

### Official Resources

- [Annual NLCD Product Suite](https://www.usgs.gov/centers/eros/science/nlcd-product-suite)
- [Annual NLCD Collection 1.2 Data Release](https://www.usgs.gov/data/annual-national-land-cover-database-nlcd-collection-1-products)
- [Annual NLCD Collection 1.2 User Guide](https://www.mrlc.gov/sites/default/files/docs/LSDS-2103%20Annual%20National%20Land%20Cover%20Database%20%28NLCD%29%20Collection%201%20Science%20Product%20User%20Guide%20-v1.2%202026_04_21.pdf)
- [MRLC Data Services](https://www.mrlc.gov/data-services-page)
- [Fractional Impervious Surface](https://www.mrlc.gov/data/type/fractional-impervious-surface)
- [Impervious Descriptor](https://www.mrlc.gov/data/type/impervious-descriptor)
